# Sprint 2: Descarga de images_002 y images_003

Descarga los dos tarballs adicionales del NIH a Drive para ampliar el dataset
de ~5k a ~15k imagenes.

**Tiempo estimado:** 40-80 min (depende del ancho de banda de Colab ese dia).
**Espacio necesario:** ~7.5 GB libres en Drive.
**Runtime:** CPU (la descarga no necesita GPU).

El notebook detecta automaticamente si un archivo ya fue descargado y lo saltea.


## 1. Montar Drive y verificar espacio

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/Tesis_CXR'
NIH_DIR = f'{PROJECT_ROOT}/data/raw/nih'
NIH_IMAGES_DIR = f'{NIH_DIR}/images'

import os, shutil
from pathlib import Path

total, used, free = shutil.disk_usage('/content/drive/MyDrive')
print(f'Drive - Total: {total/(1024**3):.1f} GB | Usado: {used/(1024**3):.1f} GB | Libre: {free/(1024**3):.1f} GB')

if free < 8 * (1024**3):
    print('\n⚠ ADVERTENCIA: menos de 8 GB libres.')
    print('  Necesitas ~7.5 GB para los dos tarballs + descompresion.')
    print('  Libera espacio antes de continuar.')
else:
    print('\n✓ Espacio suficiente.')

# Ver imagenes actuales
n_imgs = len([f for f in os.listdir(NIH_IMAGES_DIR) if f.endswith('.png')])
print(f'\nImagenes ya en Drive: {n_imgs:,} (de images_001)')


## 2. Descargar `images_002.tar.gz` (~3.7 GB)

In [ ]:
BASE_URL = 'https://archive.org/download/ChestXray-NIHCC/ChestXray-NIHCC'
EXPECTED_SIZE = {
    'images_002.tar.gz': 3_700_000_000,
    'images_003.tar.gz': 3_700_000_000,
}

def download_tarball(filename: str) -> bool:
    url = f'{BASE_URL}/{filename}'
    tar_path = f'{NIH_DIR}/{filename}'
    expected = EXPECTED_SIZE.get(filename, 3_500_000_000)

    if os.path.exists(tar_path):
        size = os.path.getsize(tar_path)
        if size > expected * 0.95:
            print(f'✓ {filename} ya existe ({size/(1024**3):.2f} GB). Saltando.')
            return True
        else:
            print(f'⚠ {filename} incompleto ({size/(1024**3):.2f} GB). Re-descargando...')
            os.remove(tar_path)

    print(f'Descargando {filename} desde Archive.org...')
    print(f'URL: {url}')
    os.system(f'wget -c --progress=bar:force:noscroll -O "{tar_path}" "{url}"')

    if os.path.exists(tar_path):
        size = os.path.getsize(tar_path)
        print(f'\n✓ Descargado ({size/(1024**3):.2f} GB): {tar_path}')
        return True
    else:
        print(f'\n✗ Error: no se encontro el archivo descargado.')
        return False

ok_002 = download_tarball('images_002.tar.gz')


## 3. Descargar `images_003.tar.gz` (~3.7 GB)

In [ ]:
ok_003 = download_tarball('images_003.tar.gz')


## 4. Verificar integridad de los dos tarballs

In [ ]:
import tarfile

for filename in ['images_002.tar.gz', 'images_003.tar.gz']:
    tar_path = f'{NIH_DIR}/{filename}'
    if not os.path.exists(tar_path):
        print(f'✗ {filename}: no encontrado')
        continue
    try:
        with tarfile.open(tar_path, 'r:gz') as tar:
            members = tar.getmembers()
        n_files = sum(1 for m in members if m.isfile())
        print(f'✓ {filename}: {n_files} archivos (integro)')
    except Exception as e:
        print(f'✗ {filename}: CORRUPTO - {e}')


## 5. Descomprimir `images_002.tar.gz`

In [ ]:
from tqdm import tqdm

def extract_tarball(filename: str, min_expected: int = 4000) -> int:
    tar_path = f'{NIH_DIR}/{filename}'
    if not os.path.exists(tar_path):
        print(f'✗ {filename} no encontrado. Saltando extraccion.')
        return 0

    # Contar imagenes actuales
    current_imgs = set(f for f in os.listdir(NIH_IMAGES_DIR) if f.endswith('.png'))
    print(f'Imagenes antes de extraer: {len(current_imgs):,}')

    print(f'Extrayendo {filename}...')
    with tarfile.open(tar_path, 'r:gz') as tar:
        members = [m for m in tar.getmembers() if m.isfile()]
        # Solo extraer las que no existen ya
        to_extract = [m for m in members
                      if os.path.basename(m.name) not in current_imgs]
        print(f'  Archivos en tar: {len(members)} | Nuevos a extraer: {len(to_extract)}')

        if to_extract:
            for m in tqdm(to_extract, desc=f'Extrayendo {filename}'):
                tar.extract(m, path=NIH_DIR)
        else:
            print('  Todas las imagenes ya existen. Saltando.')

    new_total = len([f for f in os.listdir(NIH_IMAGES_DIR) if f.endswith('.png')])
    added = new_total - len(current_imgs)
    print(f'✓ Extraccion completa. Nuevas imagenes: +{added:,} | Total: {new_total:,}')
    return new_total

total_after_002 = extract_tarball('images_002.tar.gz')


## 6. Descomprimir `images_003.tar.gz`

In [ ]:
total_after_003 = extract_tarball('images_003.tar.gz')


## 7. Actualizar CSV local con las nuevas imagenes

In [ ]:
import pandas as pd

# CSV original con todas las etiquetas (112k filas)
full_csv_path = f'{NIH_DIR}/Data_Entry_2017.csv'
df_full = pd.read_csv(full_csv_path)
print(f'CSV completo: {len(df_full):,} filas')

# Imagenes que tenemos ahora (los 3 tarballs)
all_imgs = set(f for f in os.listdir(NIH_IMAGES_DIR) if f.endswith('.png'))
print(f'Imagenes en disco (3 tarballs): {len(all_imgs):,}')

# Filtrar CSV a las imagenes disponibles
df_subset = df_full[df_full['Image Index'].isin(all_imgs)].copy()
print(f'Filas en CSV que matchean: {len(df_subset):,}')

# Guardar CSV actualizado
subset_csv_path = f'{NIH_DIR}/Data_Entry_subset_local.csv'
df_subset.to_csv(subset_csv_path, index=False)
print(f'\n✓ CSV actualizado guardado: {subset_csv_path}')
print(f'  Filas: {len(df_subset):,}')


## 8. Distribucion de clases en el nuevo subset

In [ ]:
# Ver distribucion de las 3 clases del Sprint 2
print('DISTRIBUCION EN EL SUBSET COMBINADO (3 tarballs):')
print('=' * 60)

n_no_finding = (df_subset['Finding Labels'] == 'No Finding').sum()
n_cardio = df_subset['Finding Labels'].str.contains('Cardiomegaly').sum()
n_effusion = df_subset['Finding Labels'].str.contains('Effusion').sum()
n_cardio_pure = (df_subset['Finding Labels'] == 'Cardiomegaly').sum()
n_effusion_pure = (df_subset['Finding Labels'] == 'Effusion').sum()

print(f'No Finding:                     {n_no_finding:>6,}')
print(f'Cardiomegaly (sola o combinada):{n_cardio:>6,}')
print(f'  - Cardiomegaly pura:          {n_cardio_pure:>6,}')
print(f'Effusion (sola o combinada):    {n_effusion:>6,}')
print(f'  - Effusion pura:              {n_effusion_pure:>6,}')
print()

# Estimar dataset balanceado para Sprint 2
# Clase mas chica de las 3 determina el techo
# Estrategia: ratio 2:1 entre NF y cada patologia
min_positives = min(n_cardio, n_effusion)
print(f'Clase positiva mas chica: {min_positives:,}')
print(f'Estimacion dataset Sprint 2 (1 img/paciente, ratio 2:1):')
print(f'  ~{min_positives} Cardiomegaly + ~{min_positives} Effusion + ~{min_positives*2} No Finding')
print(f'  Total estimado: ~{min_positives*4} imagenes')
print()
print('Siguiente paso: volver al chat para recibir sprint2.yaml y prepare_splits_sprint2.py')


## 9. (Opcional) Borrar tarballs para liberar espacio

In [ ]:
# Descomentar si quieres recuperar ~7.5 GB
# for filename in ['images_002.tar.gz', 'images_003.tar.gz']:
#     tar_path = f'{NIH_DIR}/{filename}'
#     if os.path.exists(tar_path):
#         os.remove(tar_path)
#         print(f'✓ Borrado: {tar_path}')
